In [1]:
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/machine-learning-comp-432-project/sample_submission.csv
/kaggle/input/machine-learning-comp-432-project/train.csv
/kaggle/input/machine-learning-comp-432-project/test.csv


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

# ============================================================
# Load Data
# ============================================================

train_path = "/kaggle/input/machine-learning-comp-432-project/train.csv"
test_path  = "/kaggle/input/machine-learning-comp-432-project/test.csv"

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# Features
feature_cols = [c for c in train_df.columns if c.startswith("feature")]

X = train_df[feature_cols].values
y = train_df["label"].values
X_test = test_df[feature_cols].values

NUM_FEATURES = X.shape[1]
NUM_CLASSES = 50 # no of unique labels

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# ============================================================
# Dataset Class
# ============================================================

class FeatureDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = None if y is None else torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]


full_dataset = FeatureDataset(X, y)

# ============================================================
# MLP MODEL
# ============================================================

class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.layers(x)


# ============================================================
# K-FOLD TRAINING
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 20
k_folds = 5

kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
fold_accuracies = []
best_model_state = None
best_fold_acc = 0

print("\nSTARTING K-FOLD TRAINING")

for fold, (train_idx, val_idx) in enumerate(kf.split(full_dataset)):
    print(f"\nFold {fold+1} / {k_folds} \n")

    train_subset = Subset(full_dataset, train_idx)
    val_subset   = Subset(full_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=256, shuffle=True)
    val_loader   = DataLoader(val_subset, batch_size=256, shuffle=False)

    # Reinitialize model each fold
    model = MLP(NUM_FEATURES, NUM_CLASSES).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

    # TRAIN
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Fold {fold+1} | Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f}")

    # VALIDATION (ONCE PER FOLD)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            preds = outputs.argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    fold_acc = correct / total
    fold_accuracies.append(fold_acc)

    print(f"Fold {fold+1} Validation Accuracy: {fold_acc:.4f}")

    # Track best model
    if fold_acc > best_fold_acc:
        best_fold_acc = fold_acc
        best_model_state = model.state_dict()

print("\nK-FOLD ACCURACIES:", fold_accuracies)
print(" BEST FOLD ACCURACY:", best_fold_acc)

# Save the best model
torch.save(best_model_state, "best_model.pth")

# ============================================================
# FINAL TEST PREDICTION
# ============================================================

# Reload best model
model = MLP(NUM_FEATURES, NUM_CLASSES).to(device)
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Load test features
test_df = pd.read_csv(test_path)
test_ids = test_df["id"]

# Scale features
X_test = test_df.drop(columns=["id"]).values
X_test = scaler.transform(X_test)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)

# Predict
with torch.no_grad():
    outputs = model(X_test_tensor)
    preds = outputs.argmax(1).cpu().numpy()

# Create submission file
submission = pd.DataFrame({
    "id": test_ids,
    "label": preds
})

submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)

print("Submission file saved to:", submission_path)
submission.head()

Train: (115406, 502)
Test : (49460, 501)

STARTING K-FOLD TRAINING

Fold 1 / 5 

Fold 1 | Epoch 1/20 | Loss: 2.8470
Fold 1 | Epoch 2/20 | Loss: 2.0418
Fold 1 | Epoch 3/20 | Loss: 1.6214
Fold 1 | Epoch 4/20 | Loss: 1.3100
Fold 1 | Epoch 5/20 | Loss: 1.0614
Fold 1 | Epoch 6/20 | Loss: 0.8629
Fold 1 | Epoch 7/20 | Loss: 0.7090
Fold 1 | Epoch 8/20 | Loss: 0.5987
Fold 1 | Epoch 9/20 | Loss: 0.5046
Fold 1 | Epoch 10/20 | Loss: 0.4390
Fold 1 | Epoch 11/20 | Loss: 0.3896
Fold 1 | Epoch 12/20 | Loss: 0.3501
Fold 1 | Epoch 13/20 | Loss: 0.3118
Fold 1 | Epoch 14/20 | Loss: 0.2920
Fold 1 | Epoch 15/20 | Loss: 0.2686
Fold 1 | Epoch 16/20 | Loss: 0.2480
Fold 1 | Epoch 17/20 | Loss: 0.2351
Fold 1 | Epoch 18/20 | Loss: 0.2244
Fold 1 | Epoch 19/20 | Loss: 0.2206
Fold 1 | Epoch 20/20 | Loss: 0.2015
Fold 1 Validation Accuracy: 0.6157

Fold 2 / 5 

Fold 2 | Epoch 1/20 | Loss: 2.8359
Fold 2 | Epoch 2/20 | Loss: 2.0358
Fold 2 | Epoch 3/20 | Loss: 1.6168
Fold 2 | Epoch 4/20 | Loss: 1.2999
Fold 2 | Epoch 5/20

,id,label
0,0,4
1,1,44
2,2,3
3,3,15
4,4,22
